# Taller B1-T1 · Quantfolio

**Toolkit de extracción multifuente, análisis bursátil y gestión de carteras con simulación de Monte Carlo.**

Este cuaderno es el punto de entrada reproducible del taller: recorre la ingesta de datos (acciones, índices,
criptomonedas y macroeconomía), el modelado de series y carteras, los informes y la simulación Monte Carlo.

Requisitos: haber ejecutado `uv sync` y disponer de conexión a internet (la suite de pruebas no la necesita).

In [ ]:
from quantfolio import (
    CachedPriceSource,
    FredMacroSource,
    KrakenSource,
    MonteCarloParameters,
    Portfolio,
    YahooFinanceSource,
)

START = "2025-01-02"

## 1 · Extracción multifuente: acciones e índices (Yahoo Finance)

`get_many()` descarga varias series en lote a partir de una lista de activos. Cada serie se normaliza en un objeto
`PriceSeries` con su ticker, su origen, sus estadísticos automáticos y sus precios limpios.

In [ ]:
yahoo = YahooFinanceSource()

equities = yahoo.get_many(["META", "GOOGL", "IAG.MC", "MC.PA"], start=START)
indices = yahoo.get_many(["^GSPC", "^IBEX"], start=START)

for ticker, asset in {**equities, **indices}.items():
    print(
        f"{ticker:>8} | {asset.source:<13} | "
        f"{asset.prices.index.min().date()} → {asset.prices.index.max().date()} | "
        f"{len(asset.prices)} observaciones"
    )

## 2 · Una segunda API: Kraken

Kraken expone una API pública OHLC sin clave. Su adaptador devuelve **exactamente el mismo objeto interno** que
Yahoo Finance: la capa de análisis es independiente de la API consultada.

In [ ]:
kraken = KrakenSource()
btc = kraken.get_prices("BTC/USD", start=START)

print(f"{type(btc).__name__}: {btc.ticker} | {btc.source} | {len(btc.prices)} observaciones")
print(f"Media: {btc.mean_price:,.0f} USD | Desviación típica: {btc.std_price:,.0f} USD")
print(f"¿Mismo objeto interno que las acciones? {type(btc) is type(equities['META'])}")

## 3 · Tipología adicional: tipos de interés (FRED)

`FredMacroSource` descarga indicadores macroeconómicos del export CSV sin clave de FRED y los normaliza en
`MacroSeries`. Aquí se usa el rendimiento del bono estadounidense a 10 años (`DGS10`).

In [ ]:
fred = FredMacroSource()
rates = fred.get_series("DGS10", start=START)

print(f"{type(rates).__name__}: {rates.series_id} | {rates.source}")
print(f"Últimos valores (%): {rates.values.tail(3).round(2).to_list()}")

## 4 · Caché CSV local

`CachedPriceSource` envuelve a cualquier proveedor y guarda cada serie en `data/<proveedor>/<ticker>.csv`. El
archivo se reutiliza mientras cubra el intervalo solicitado y su último dato siga vigente; si no, se descarga y se
reemplaza.

In [ ]:
cached = CachedPriceSource(YahooFinanceSource(), cache_dir="data")

first = cached.get_prices("META", start=START)
second = cached.get_prices("META", start=START)

print("Origen de la primera llamada:", first.source)
print("Origen de la segunda llamada:", second.source)

## 5 · La cartera

Una cartera es una colección de activos junto con la fracción del capital invertido en cada uno. `Portfolio`
valida las ponderaciones (finitas, no negativas, con clave para cada activo) y las **normaliza** para que sumen 1;
después alinea los calendarios bursátiles por intersección y calcula las métricas ponderadas.

In [ ]:
portfolio = Portfolio(
    assets=cached.get_many(["META", "GOOGL", "IAG.MC", "MC.PA"], start=START),
    weights={"META": 0.35, "GOOGL": 0.25, "IAG.MC": 0.20, "MC.PA": 0.20},
)

print("Pesos normalizados:")
for ticker, weight in portfolio.weights.items():
    print(f"  {ticker}: {weight:.2%}")

print("\nDiagnóstico de alineación temporal:", portfolio.alignment_summary())

print("\nEstadísticos anualizados:")
for name, value in portfolio.summary().items():
    print(f"  {name}: {value:.6f}")

## 6 · Informe Markdown parametrizable

`report()` genera composición, métricas, calidad de los datos y advertencias de riesgo derivadas del estado real
de la cartera (cobertura de alineación, concentración y longitud de la muestra).

In [ ]:
from IPython.display import Markdown

Markdown(portfolio.report())

In [ ]:
# El informe es parametrizable: se pueden omitir secciones o cambiar la anualización.
print(portfolio.report(include_warnings=False))

## 7 · Panel analítico de visualizaciones

`plots_report()` muestra la evolución de precios, la rentabilidad acumulada por activo, la matriz de correlación y
la evolución del valor de la cartera (base 100).

In [ ]:
portfolio.plots_report()

## 8 · Monte Carlo de la cartera

La simulación modela la cartera con un movimiento browniano geométrico parametrizable
(`MonteCarloParameters`): número de trayectorias, horizonte, valor inicial, supuestos de rentabilidad y volatilidad,
y semilla. Por defecto, los supuestos se estiman de las rentabilidades históricas.

In [ ]:
simulations = portfolio.simulate_monte_carlo(
    MonteCarloParameters(n_paths=2_000, horizon=252, seed=42)
)

final_values = simulations.iloc[-1]
print(f"Valor inicial: {simulations.iloc[0, 0]:,.0f}")
print(final_values.describe(percentiles=[0.05, 0.5, 0.95]).round(2))

In [ ]:
portfolio.plot_simulation(simulations, max_paths=200)

## 9 · Monte Carlo de un activo con supuestos propios

La misma simulación está disponible en `PriceSeries`. Aquí se simula el S&P 500 asumiendo un 8 % de rentabilidad
anual y un 18 % de volatilidad, en lugar de los valores históricos, y partiendo del último cierre observado.

In [ ]:
scenario = indices["^GSPC"].simulate_monte_carlo(
    MonteCarloParameters(
        n_paths=1_000,
        horizon=252,
        return_assumption=0.08,
        volatility_assumption=0.18,
        seed=7,
    )
)

print(f"Punto de partida: {scenario.iloc[0, 0]:,.0f} (último cierre de ^GSPC)")
print(f"Mediana a un año: {scenario.iloc[-1].median():,.0f}")

In [ ]:
indices["^GSPC"].plot_simulation(scenario, max_paths=150)

## 10 · Calidad del código y siguientes pasos

Los controles de calidad del proyecto se ejecutan con:

```console
uv run ruff check .
uv run ruff format --check .
uv run mypy src tests
uv run pytest
```

- La biblioteca vive en `src/quantfolio/` (solo código importable y probado); este cuaderno y la caché `data/` no
  forman parte del paquete.
- Diagrama de arquitectura UML y de flujo: [`docs/architecture.md`](../docs/architecture.md).
- Correspondencia completa con el enunciado del taller: README del repositorio.